# Phase 4C.2H.0 — development diagnostics and protocol lock

Thin launcher for the bounded 8-source development smoke and deterministic fold commitments. It does not launch the 150-fit official nested-CV experiment.

In [ ]:
import csv, hashlib, tarfile
from pathlib import Path, PurePosixPath
from google.colab import drive
drive.mount('/content/drive')

RUN_DEVELOPMENT_SMOKE = True
REPO = Path('/content/forensics-web-lab')
BUNDLE_ARCHIVE = Path('/content/drive/MyDrive/Colab Notebooks/forensics-web-lab/phase_4c1/inputs/phase_4c1_binary_n250_reusable.tar')
BUNDLE = Path('/content/phase4c2h-development-bundle')
OUTPUT = Path('/content/drive/MyDrive/Colab Notebooks/forensics-web-lab/phase_4c2h/development_diagnostics.json')
EXPECTED_ARCHIVE_SHA256 = 'd49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27'
EXPECTED_MANIFEST_SHA256 = '411e35da80843a10f7bb22e33efac8a4d1db70110dbda91cadbbf39532c9312d'
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()
assert sha256_file(BUNDLE_ARCHIVE) == EXPECTED_ARCHIVE_SHA256
manifest = BUNDLE / 'manifest_pilot_a_option_p.csv'
if not manifest.exists():
    BUNDLE.mkdir(parents=True, exist_ok=False)
    with tarfile.open(BUNDLE_ARCHIVE, 'r') as archive:
        members = archive.getmembers()
        assert all(m.isfile() and not PurePosixPath(m.name).is_absolute() and '..' not in PurePosixPath(m.name).parts for m in members)
        archive.extractall(BUNDLE, members=members)
assert sha256_file(manifest) == EXPECTED_MANIFEST_SHA256
with manifest.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 341 and {r['partition'] for r in rows} == {'development_train', 'inner_validation'}


In [ ]:
import subprocess, sys
command = [
    sys.executable, str(REPO / 'scripts/research/run_phase_4c2h_development_diagnostics.py'),
    '--protocol', str(REPO / 'ml/configs/phase_4c2h_development_nested_cv.yaml'),
    '--manifest', str(BUNDLE / 'manifest_pilot_a_option_p.csv'),
    '--data-root', str(BUNDLE), '--output', str(OUTPUT),
    '--smoke-sources', '8', '--smoke-steps', '300', '--seed', '42',
]
if RUN_DEVELOPMENT_SMOKE:
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(command, cwd=REPO, check=True)
else:
    print('Development smoke disabled; no model execution performed.')


In [ ]:
import json
receipt = json.loads(OUTPUT.read_text())
print(json.dumps({
    'planned_training_budget': receipt['planned_training_budget'],
    'diagnostic_passes': {key: value.get('pass') for key, value in receipt['diagnostics'].items() if isinstance(value, dict) and 'pass' in value},
}, indent=2))
